In [ ]:
# 🧠 AI Mental Health Sentiment Analyzer
# Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

nltk.download('stopwords')

print("✅ Libraries imported successfully!")


In [ ]:
# Load Dataset
# For testing (sample dataset first)

data = {
    "text": [
        "I feel happy and motivated today!",
        "Life seems meaningless. I’m so tired of everything.",
        "Had a productive day, feeling great.",
        "I feel so alone and depressed.",
        "Things are getting better. I'm hopeful!",
        "I don't want to talk to anyone.",
        "My anxiety is getting worse.",
        "I love spending time with my family.",
        "I feel confident and at peace.",
        "I can't stop crying."
    ],
    "sentiment": [
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "negative",
        "positive",
        "positive",
        "negative"
    ]
}

df = pd.DataFrame(data)
df.head()


In [ ]:
# Preprocess Text Data

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove links
    text = re.sub(r'\@w+|\#','', text)                  # remove mentions/hashtags
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\d+', '', text)                     # remove numbers
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

df['clean_text'] = df['text'].apply(clean_text)
df.head()


In [ ]:
# Split Data

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])


In [ ]:
# Convert Text to Numerical Features using TF-IDF

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("✅ TF-IDF vectorization complete!")


In [ ]:
# Train the Sentiment Model

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

print("✅ Model training complete!")


In [ ]:
# Evaluate the Model

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# Save the Model and Vectorizer

joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("✅ Model and vectorizer saved successfully!")


In [ ]:
# Test the Model with New Inputs

def predict_sentiment(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return prediction

# Example
sample_text = "I feel so anxious and alone."
print(f"Input: {sample_text}")
print(f"Predicted Sentiment: {predict_sentiment(sample_text)}")
